# Step 7 (optional): Ablation
Which part helps: attention only, weighted fusion only, or both? Each extra run costs one full training. Set `RUN_ABLATION = True` to run.

In [ ]:
# Set True for a fast CPU sanity run (3 epochs, 320 px, 25% of train). False = full experiment.
QUICK_TEST = False
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # works from VS Code either way
sys.path.insert(0, str(ROOT / "src"))
import mri_detection as md
from mri_detection import data, viz, models, train, evaluate
cfg = md.load_config(quick=QUICK_TEST)
RUNS = md.runs_dir(cfg)
YOLO_DIR = md.ROOT / cfg["dataset"]["yolo_dir"]
DATA_YAML = data.training_yaml(cfg, YOLO_DIR)
import yaml, pandas as pd
from IPython.display import display
names = list(yaml.safe_load(open(DATA_YAML))["names"].values())
IMGSZ, BATCH = cfg["train"]["imgsz"], cfg["train"]["batch"]
DEVICE = md.config.resolve_device(cfg["train"]["device"])
RUN_ABLATION = False

In [ ]:
if RUN_ABLATION:
    BEST_A = train.train_improved(cfg, DATA_YAML, RUNS, "ablation_attention_only", len(names), attention=True, weighted=False)
    BEST_B = train.train_improved(cfg, DATA_YAML, RUNS, "ablation_fusion_only", len(names), attention=False, weighted=True)
    rows = {}
    for label, w in [("Baseline", RUNS / "baseline_yolov8m/weights/best.pt"), ("+ Attention only", BEST_A),
                     ("+ Weighted fusion only", BEST_B), ("+ Both (proposed)", RUNS / "proposed_yolov8m/weights/best.pt")]:
        s, _ = evaluate.evaluate(w, label.replace(" ", "_").replace("+", ""), DATA_YAML, RUNS, IMGSZ, BATCH, "val", DEVICE)
        rows[label] = s
    abl = pd.DataFrame(rows).T
    display(abl.round(4)); abl.round(4).to_csv(RUNS / "ablation_val.csv")